# Sesión 01 — Repaso de Probabilidad e Introducción al Curso
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo I · Fundamentos del Aprendizaje Estadístico**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Caracterizar una variable aleatoria mediante su PDF/CDF, media y varianza.
2. Aplicar la ley de probabilidad total y el teorema de Bayes en contextos biomédicos.
3. Derivar estimadores de máxima verosimilitud para familias paramétricas simples.
4. Reconocer dónde aparecen estos conceptos en el procesamiento de señales biomédicas y en la toma de decisiones clínicas.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Bishop, C.M. (2006). *Pattern Recognition and Machine Learning*. Springer. **Capítulo 1** (§1.1–1.2, §1.4). |
| ★★★ | Murphy, K.P. (2022). *Probabilistic Machine Learning: An Introduction*. MIT Press. **Capítulo 2** (acceso abierto: probml.ai). |
| ★★☆ | Theodoridis, S. & Koutroumbas, K. (2008). *Pattern Recognition* (4ª ed.). Academic Press. **Capítulo 1**. |
| ★☆☆ | VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly. **Capítulo 4** (repaso de NumPy/matplotlib). |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# Estilo de graficación consistente para todo el curso
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Configuración completa.')

## Parte 1 — Variables aleatorias, PDF y CDF

Una **variable aleatoria** $X$ asigna valores reales a los resultados de un experimento aleatorio.  
Su **función de densidad de probabilidad** (PDF) satisface $p(x) \ge 0$ e $\int_{-\infty}^{\infty} p(x)\,dx = 1$.  
La **función de distribución acumulada** (CDF) es $F(x) = P(X \le x) = \int_{-\infty}^{x} p(t)\,dt$.

### 1.1 Distribución gaussiana — el caballo de batalla del modelado de señales biomédicas

$$p(x \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

In [ ]:
x = np.linspace(-5, 5, 500)

parametros = [
    (0, 1,   'μ=0, σ=1  (estándar)'),
    (1, 0.5, 'μ=1, σ=0.5'),
    (-1, 2,  'μ=-1, σ=2'),
]

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

for mu, sigma, etiqueta in parametros:
    dist = stats.norm(mu, sigma)
    ejes[0].plot(x, dist.pdf(x), lw=2, label=etiqueta)
    ejes[1].plot(x, dist.cdf(x), lw=2, label=etiqueta)

ejes[0].set(title='PDF  $p(x)$', xlabel='x', ylabel='densidad')
ejes[1].set(title='CDF  $F(x) = P(X \\leq x)$', xlabel='x', ylabel='probabilidad')
ejes[0].legend(fontsize=9)
fig.suptitle('Familia gaussiana', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 1.2 Estadística descriptiva y la distribución empírica

**Contexto biomédico:** La frecuencia cardiaca en reposo de un adulto sano sigue aproximadamente una distribución gaussiana.  
Simulemos una cohorte y recuperemos sus parámetros.

In [ ]:
# Simular FC en reposo (lpm) para N=200 sujetos
N = 200
fc_mu_real, fc_sigma_real = 70.0, 10.0
fc_muestras = rng.normal(fc_mu_real, fc_sigma_real, N)

# Estimaciones empíricas
emp_mu    = np.mean(fc_muestras)
emp_sigma = np.std(fc_muestras, ddof=1)   # sin sesgo

print(f'Parámetros reales   : μ = {fc_mu_real:.1f} lpm,  σ = {fc_sigma_real:.1f} lpm')
print(f'Estimaciones muestrales: μ̂ = {emp_mu:.2f} lpm, σ̂ = {emp_sigma:.2f} lpm')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(fc_muestras, bins=25, density=True, alpha=0.6, color='steelblue', label='Empírica')
xr = np.linspace(fc_muestras.min()-5, fc_muestras.max()+5, 300)
ax.plot(xr, stats.norm(fc_mu_real, fc_sigma_real).pdf(xr), 'r--', lw=2, label='Real')
ax.plot(xr, stats.norm(emp_mu, emp_sigma).pdf(xr),         'k-',  lw=2, label='Ajuste EMV')
ax.set(xlabel='Frecuencia cardiaca (lpm)', ylabel='Densidad',
       title='FC en reposo — cohorte simulada')
ax.legend()
plt.tight_layout()
plt.show()

## Parte 2 — Estimación de Máxima Verosimilitud (EMV)

Dadas $N$ muestras i.i.d. $\mathcal{D} = \{x_1, \dots, x_N\}$ de $p(x \mid \theta)$, la EMV busca:

$$\hat{\theta}_{\text{EMV}} = \arg\max_\theta \log p(\mathcal{D} \mid \theta) = \arg\max_\theta \sum_{i=1}^{N} \log p(x_i \mid \theta)$$

**Para la gaussiana**, las soluciones son la media muestral y la varianza (sesgada).

### 2.1 Visualización de la superficie de log-verosimilitud

In [ ]:
mu_grid    = np.linspace(55, 85, 200)
sigma_grid = np.linspace(5,  20, 200)
MU, SG     = np.meshgrid(mu_grid, sigma_grid)

def log_verosimilitud_gaussiana(datos, mu, sigma):
    N = len(datos)
    return (-N/2)*np.log(2*np.pi*sigma**2) - np.sum((datos - mu)**2)/(2*sigma**2)

LL = np.array([
    [log_verosimilitud_gaussiana(fc_muestras, MU[i,j], SG[i,j]) for j in range(200)]
    for i in range(200)
])

fig, ax = plt.subplots(figsize=(8, 5))
cf = ax.contourf(MU, SG, LL, levels=40, cmap='viridis')
plt.colorbar(cf, ax=ax, label='Log-verosimilitud')
ax.axvline(emp_mu,    color='w', lw=1.5, ls='--', label=f'μ̂ = {emp_mu:.1f}')
ax.axhline(emp_sigma, color='w', lw=1.5, ls=':',  label=f'σ̂ = {emp_sigma:.1f}')
ax.scatter([emp_mu], [emp_sigma], c='red', s=80, zorder=5, label='EMV')
ax.set(xlabel='μ (lpm)', ylabel='σ (lpm)',
       title='Superficie de log-verosimilitud $\\mathcal{L}(\\mu,\\sigma)$')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print('El punto rojo está en el máximo — coincide con la media y desviación estándar muestrales.')

## Parte 3 — Gaussiana multivariada y estructura de covarianza

Las señales biomédicas raramente son univariadas. Los canales EEG, características fisiológicas
o vóxeles de imágenes forman **vectores aleatorios multivariados** $\mathbf{x} \in \mathbb{R}^d$.

$$p(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma}) = \frac{1}{(2\pi)^{d/2}|\boldsymbol{\Sigma}|^{1/2}} \exp\!\left(-\tfrac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

In [ ]:
# Simular dos características fisiológicas correlacionadas:
# Característica 1: FC (lpm), Característica 2: PAS (mmHg)
mu_bio    = np.array([70, 120])
Sigma_bio = np.array([[100, 60],
                       [60,  225]])   # correlación positiva

X_bio = rng.multivariate_normal(mu_bio, Sigma_bio, 300)

# Rejilla para la gráfica de contorno
g1 = np.linspace(30, 110, 200)
g2 = np.linspace(70, 175, 200)
G1, G2 = np.meshgrid(g1, g2)
pos = np.dstack([G1, G2])
rv  = stats.multivariate_normal(mu_bio, Sigma_bio)
Z   = rv.pdf(pos)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(G1, G2, Z, levels=12, cmap='Blues', alpha=0.7)
ax.contour( G1, G2, Z, levels=12, colors='royalblue', linewidths=0.6)
ax.scatter(X_bio[:,0], X_bio[:,1], alpha=0.3, s=12, color='navy')
ax.set(xlabel='Frecuencia cardiaca (lpm)', ylabel='Presión arterial sistólica (mmHg)',
       title='Gaussiana bivariada — FC vs PAS')
plt.tight_layout()
plt.show()

rho = Sigma_bio[0,1] / np.sqrt(Sigma_bio[0,0]*Sigma_bio[1,1])
print(f'Correlación de Pearson real : ρ = {rho:.2f}')
print(f'Correlación muestral        : ρ̂ = {np.corrcoef(X_bio.T)[0,1]:.2f}')

## Parte 3b — Reglas de la suma y del producto (estilo Fig. 1.11 de Bishop)

Antes de pasar a las densidades continuas, Bishop visualiza los fundamentos de la
probabilidad con dos variables **discretas** en una cuadrícula joint/marginal.
La idea es sencilla pero poderosa:

| Regla | Fórmula | Significado |
|---|---|---|
| **Suma** | $p(X=x) = \sum_y p(X=x, Y=y)$ | La marginal se obtiene sumando sobre la otra variable |
| **Producto** | $p(X, Y) = p(Y \mid X)\,p(X)$ | La conjunta = condicional × marginal |
| **Bayes** | $p(Y \mid X) = p(X \mid Y)\,p(Y)\,/\,p(X)$ | Aplicar la regla del producto dos veces |

Reproducimos la figura con variables biomédicas concretas:
- $X$ = nivel de glucosa en ayunas (4 rangos discretos, en mg/dL)
- $Y$ = diagnóstico clínico (Normal / Prediabetes / Diabetes)

La cuadrícula central es $p(X, Y)$ — cada celda es la probabilidad conjunta.
Sumar las filas da $p(Y)$ (marginal del diagnóstico).
Sumar las columnas da $p(X)$ (marginal de glucosa).
Una columna normalizada es $p(Y \mid X=x)$ — la probabilidad del diagnóstico dado el nivel de glucosa.

> **Fuente de los datos:** INSP / Secretaría de Salud (2023). *Encuesta Nacional de Salud y Nutrición 2022* (ENSANUT 2022). Instituto Nacional de Salud Pública, México. https://ensanut.insp.mx/encuestas/ensanut2022/doctos/informes/ensanut_2022_presentacion_resultados.pdf  
> Los valores de $p(X, Y)$ son aproximaciones didácticas calibradas con las prevalencias reportadas de diabetes (12.6 %) y prediabetes (22.0 %) en adultos mexicanos.

In [ ]:
# ── Distribución conjunta p(X, Y) — estilo Bishop Fig. 1.11 ──────────────────
# X = nivel de glucosa en ayunas (mg/dL)  — 4 categorías
# Y = diagnóstico clínico                 — 3 categorías

glucosa_labels    = ['<100\n(bajo)', '100–125\n(límite)', '126–160\n(alto)', '>160\n(muy alto)']
dx_labels         = ['Normal', 'Prediabetes', 'Diabetes']

# p(X, Y) — tabla conjunta (filas = diagnóstico, columnas = glucosa)
# Valores inspirados en epidemiología de DM2 en México (ENSANUT 2022)
p_joint = np.array([
    [0.38, 0.10, 0.02, 0.00],   # Normal
    [0.08, 0.14, 0.06, 0.01],   # Prediabetes
    [0.01, 0.03, 0.10, 0.07],   # Diabetes
], dtype=float)

# Verificar que suma a 1
print(f'Suma de p(X,Y) = {p_joint.sum():.2f}  (debe ser 1.0)')

# Marginales
p_Y = p_joint.sum(axis=1)   # sumar columnas → p(Y)
p_X = p_joint.sum(axis=0)   # sumar filas    → p(X)

print('\nMarginal p(Y) — diagnóstico:')
for lbl, p in zip(dx_labels, p_Y):
    print(f'  p(Y={lbl:<12}) = {p:.2f}')

print('\nMarginal p(X) — glucosa:')
for lbl, p in zip(glucosa_labels, p_X):
    print(f'  p(X={lbl.replace(chr(10)," "):<18}) = {p:.2f}')

# Condicional p(Y|X) — normalizar cada columna
p_Y_given_X = p_joint / p_X[np.newaxis, :]   # broadcast: divide cada columna por p(X)

print('\nCondicional p(Y|X) — diagnóstico dado nivel de glucosa:')
print(f'{"":<14}', '  '.join(f'{l.replace(chr(10)," "):>18}' for l in glucosa_labels))
for i, lbl in enumerate(dx_labels):
    vals = '  '.join(f'{p_Y_given_X[i,j]:>18.2f}' for j in range(4))
    print(f'{lbl:<14} {vals}')

# ── Figura estilo Bishop ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(13, 7))
gs  = fig.add_gridspec(
    2, 3,
    width_ratios=[4, 0.8, 2.2],
    height_ratios=[3, 1],
    hspace=0.08, wspace=0.35
)

ax_joint  = fig.add_subplot(gs[0, 0])   # cuadrícula conjunta
ax_pY     = fig.add_subplot(gs[0, 1])   # marginal p(Y) — barras horizontales
ax_pX     = fig.add_subplot(gs[1, 0])   # marginal p(X) — barras verticales
ax_cond   = fig.add_subplot(gs[:, 2])   # condicional p(Y|X)

colores_dx = ['#3B82F6', '#F59E0B', '#EF4444']   # azul, ámbar, rojo

# — Cuadrícula conjunta p(X,Y) —
im = ax_joint.imshow(p_joint, cmap='Blues', aspect='auto', vmin=0, vmax=0.40)
for i in range(3):
    for j in range(4):
        val = p_joint[i, j]
        color = 'white' if val > 0.18 else '#1B3A6B'
        ax_joint.text(j, i, f'{val:.2f}', ha='center', va='center',
                      fontsize=12, fontweight='bold', color=color)

ax_joint.set_xticks(range(4))
ax_joint.set_xticklabels(glucosa_labels, fontsize=9)
ax_joint.set_yticks(range(3))
ax_joint.set_yticklabels(dx_labels, fontsize=10)
ax_joint.set_xlabel('Glucosa en ayunas $X$ (mg/dL)', fontsize=10)
ax_joint.set_ylabel('Diagnóstico $Y$', fontsize=10)
ax_joint.set_title('Distribución conjunta $p(X, Y)$\n'
                    '← sumar filas → $p(Y)$  |  ↓ sumar columnas → $p(X)$',
                    fontsize=10, pad=8)
# Bordes de celda
for i in range(3):
    for j in range(4):
        ax_joint.add_patch(plt.Rectangle(
            (j-0.5, i-0.5), 1, 1,
            fill=False, edgecolor='white', lw=2
        ))

# — Marginal p(Y) — barras horizontales a la derecha —
ax_pY.barh(range(3), p_Y, color=colores_dx, alpha=0.85, height=0.6)
for i, p in enumerate(p_Y):
    ax_pY.text(p + 0.005, i, f'{p:.2f}', va='center', fontsize=9, color='#1B3A6B')
ax_pY.set_xlim(0, 0.65)
ax_pY.set_yticks(range(3))
ax_pY.set_yticklabels([])
ax_pY.set_xlabel('$p(Y)$', fontsize=9)
ax_pY.set_title('Marginal\n$p(Y)$', fontsize=9)
ax_pY.grid(True, axis='x', alpha=0.3)

# — Marginal p(X) — barras verticales abajo —
ax_pX.bar(range(4), p_X, color='steelblue', alpha=0.7, width=0.6)
for j, p in enumerate(p_X):
    ax_pX.text(j, p + 0.003, f'{p:.2f}', ha='center', fontsize=9, color='#1B3A6B')
ax_pX.set_ylim(0, 0.60)
ax_pX.set_xticks(range(4))
ax_pX.set_xticklabels(glucosa_labels, fontsize=9)
ax_pX.set_ylabel('$p(X)$', fontsize=9)
ax_pX.set_title('Marginal $p(X)$', fontsize=9)
ax_pX.grid(True, axis='y', alpha=0.3)

# — Condicional p(Y|X) — barras apiladas —
bottom = np.zeros(4)
for i, (lbl, color) in enumerate(zip(dx_labels, colores_dx)):
    ax_cond.bar(range(4), p_Y_given_X[i], bottom=bottom,
                 color=color, alpha=0.85, width=0.55, label=lbl)
    # Anotar si la barra es suficientemente alta
    for j in range(4):
        h = p_Y_given_X[i, j]
        if h > 0.07:
            ax_cond.text(j, bottom[j] + h/2, f'{h:.2f}',
                          ha='center', va='center',
                          fontsize=9, color='white', fontweight='bold')
    bottom += p_Y_given_X[i]

ax_cond.set_xticks(range(4))
ax_cond.set_xticklabels(glucosa_labels, fontsize=9)
ax_cond.set_ylabel('Probabilidad', fontsize=9)
ax_cond.set_ylim(0, 1.08)
ax_cond.set_title('Condicional $p(Y \\mid X)$\n'
                   '(cada columna de $p(X,Y)$ normalizada)', fontsize=10)
ax_cond.legend(fontsize=9, loc='upper left')
ax_cond.axhline(1, color='gray', lw=0.8, ls='--')
ax_cond.grid(True, axis='y', alpha=0.3)

fig.suptitle(
    'Reglas de la suma y del producto — estilo Bishop Fig. 1.11\n'
    'Variables: glucosa en ayunas (X) vs diagnóstico clínico (Y)',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

# ── Verificar la regla de Bayes numéricamente ─────────────────────────────────
print('\nVerificación de la regla de Bayes: p(Y|X) = p(X|Y)·p(Y) / p(X)')
# p(X|Y) = p(X,Y) / p(Y)  → normalizar cada fila
p_X_given_Y = p_joint / p_Y[:, np.newaxis]
# Reconstruir p(Y|X) usando Bayes
p_YX_bayes = p_X_given_Y.T * p_Y / p_X[:, np.newaxis]   # (4,3)
print(f'  Diferencia máxima vs cálculo directo: {np.abs(p_YX_bayes.T - p_Y_given_X).max():.2e}  ✓')

## Parte 4 — Ley de probabilidad total y marginalización

Para una mezcla discreta de $K$ clases (por ejemplo, sanos vs pacientes):

$$p(x) = \sum_{k=1}^{K} p(x \mid C_k)\, p(C_k)$$

Ésta es la ecuación fundamental detrás de los modelos de mezcla y del clasificador Naive Bayes.

In [ ]:
# Mezcla de dos clases: sanos vs pacientes con IC (FC)
prior_S,  prior_IC  = 0.70, 0.30          # probabilidades a priori
mu_S,  sigma_S      = 70.0, 8.0           # sanos
mu_IC, sigma_IC     = 85.0, 15.0          # insuficiencia cardiaca

x_fc = np.linspace(30, 140, 600)
p_x_S  = stats.norm(mu_S,  sigma_S ).pdf(x_fc)
p_x_IC = stats.norm(mu_IC, sigma_IC).pdf(x_fc)
p_x    = prior_S * p_x_S + prior_IC * p_x_IC   # marginal

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(x_fc, prior_S  * p_x_S,  alpha=0.4, color='steelblue',
                label='Sanos   (prior=0.70)')
ax.fill_between(x_fc, prior_IC * p_x_IC, alpha=0.4, color='tomato',
                label='IC      (prior=0.30)')
ax.plot(x_fc, p_x, 'k-', lw=2, label='Marginal $p(x)$')
ax.set(xlabel='Frecuencia cardiaca (lpm)', ylabel='Densidad',
       title='Densidad marginal = suma ponderada de densidades condicionales por clase')
ax.legend()
plt.tight_layout()
plt.show()

## Parte 5 — Anticipo: ¿hacia dónde vamos?

La gráfica anterior muestra dos densidades condicionales que se superponen — exactamente el punto de partida
para la **clasificación óptima de Bayes** (Sesión 2).  
El cociente $p(C_k \mid x) = p(x \mid C_k)p(C_k)/p(x)$ es la **probabilidad posterior** que usaremos para tomar decisiones.

### 5.1 La regla de decisión de Bayes — y las DOS fronteras

Con varianzas desiguales ($\sigma_S = 8$, $\sigma_{IC} = 15$), las posteriors se cruzan **dos veces**.
Esto es un resultado de QDA: varianzas iguales producen una frontera (LDA); varianzas desiguales producen dos (QDA).

In [ ]:
# Probabilidades posteriores
post_S  = (prior_S  * p_x_S ) / p_x
post_IC = (prior_IC * p_x_IC) / p_x

# Encontrar AMBAS fronteras de decisión
# (las posteriors se cruzan donde su diferencia cambia de signo)
diff         = post_S - post_IC
cambios_signo = np.where(np.diff(np.sign(diff)))[0]
fronteras = [
    x_fc[i] + (x_fc[i+1]-x_fc[i]) * abs(diff[i]) / (abs(diff[i]) + abs(diff[i+1]))
    for i in cambios_signo
]

print(f'Número de fronteras de decisión: {len(fronteras)}')
for k, b in enumerate(fronteras):
    print(f'  Frontera {k+1}: FC ≈ {b:.1f} lpm')

fig, ejes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Panel superior: densidades escaladas
ejes[0].fill_between(x_fc, prior_S  * p_x_S,  alpha=0.4, color='steelblue', label='Sanos')
ejes[0].fill_between(x_fc, prior_IC * p_x_IC, alpha=0.4, color='tomato',    label='IC')
ejes[0].plot(x_fc, p_x, 'k-', lw=1.5, label='Marginal $p(x)$')
ejes[0].set(ylabel='Densidad',
            title='Densidades condicionales por clase (escaladas por el prior)\n'
                  'Nota: IC tiene mayor varianza (σ=15) → curva más ancha y plana')
ejes[0].legend(fontsize=9)
for b in fronteras:
    ejes[0].axvline(b, color='gray', ls='--', lw=1.2)

# Panel inferior: posteriors y regiones de decisión
ejes[1].plot(x_fc, post_S,  color='steelblue', lw=2.5, label='$P(\\text{Sano}|x)$')
ejes[1].plot(x_fc, post_IC, color='tomato',    lw=2.5, label='$P(\\text{IC}|x)$')

# Sombrear regiones de decisión
clasif = np.where(post_S >= post_IC, 0, 1)   # 0=Sano, 1=IC
ejes[1].fill_between(x_fc, 0, 1, where=(clasif==0),
                      alpha=0.08, color='steelblue', label='Clasificado: Sano')
ejes[1].fill_between(x_fc, 0, 1, where=(clasif==1),
                      alpha=0.08, color='tomato',    label='Clasificado: IC')

for k, b in enumerate(fronteras):
    ejes[1].axvline(b, color='gray', ls='--', lw=1.5,
                    label=f'Frontera {k+1}: {b:.0f} lpm')

ejes[1].set(xlabel='Frecuencia cardiaca (lpm)', ylabel='Probabilidad posterior',
            title='Probabilidades posteriores — clasificador de Bayes con DOS fronteras\n'
                  '(consecuencia de varianzas desiguales)')
ejes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print()
print('Regla de decisión (óptima según Bayes):')
if len(fronteras) == 2:
    b1, b2 = fronteras
    print(f'  FC < {b1:.0f} lpm              → Sano  (P_S domina en FC baja)')
    print(f'  {b1:.0f} < FC < {b2:.0f} lpm  → IC    (P_IC domina en la zona intermedia)')
    print(f'  FC > {b2:.0f} lpm              → Sano  (P_S vuelve a dominar en FC muy alta)')
    print()
    print('¿Por qué DOS fronteras?')
    print('  σ_IC (15 lpm) > σ_S (8 lpm): la gaussiana de IC es más ancha y plana.')
    print('  En FC muy alta, la cola de IC cae por debajo de la cola de Sanos')
    print('  porque IC ya distribuyó su masa de probabilidad más dispersamente.')
    print('  Sumado al prior mayor de Sanos (0.70 vs 0.30), la posterior de')
    print('  Sanos vuelve a ganar en la cola derecha.')
    print()
    print('Esto es el resultado clásico del Análisis Discriminante Cuadrático (QDA).')
    print('Varianzas iguales → una frontera (LDA).  Varianzas desiguales → dos (QDA).')

## ✏️ Ejercicios

1. **EMV para la Exponencial.** El intervalo entre latidos (RR) suele modelarse como una variable aleatoria exponencial con tasa $\lambda$.  
   (a) Escribe la log-verosimilitud $\log p(\mathcal{D}\mid\lambda)$ para $N$ observaciones i.i.d.  
   (b) Deriva $\hat{\lambda}_{\text{EMV}}$ analíticamente.  
   (c) Simula 150 muestras de intervalo RR de $\text{Exp}(\lambda=1.2)$ y verifica tu fórmula.

2. **Estructura de covarianza.** Genera tres conjuntos de datos gaussianos multivariados con $\boldsymbol{\mu} = \mathbf{0}$ pero distintas matrices de covarianza: (i) identidad, (ii) correlación positiva $\rho=0.9$, (iii) correlación negativa $\rho=-0.7$. Grafica los contornos y explica qué indica la forma de las elipses sobre las características.

3. **Descomposición de mezcla.** Se ha reportado que la potencia de la banda alfa del EEG en reposo presenta distribución bimodal entre sujetos, lo que sugiere una mezcla de dos subpoblaciones. Ajusta una mezcla gaussiana de dos componentes al arreglo `alfa_potencia` que se define a continuación, usando la fórmula de la verosimilitud marginal (sin EM — solo exploración visual).

```python
alfa_potencia = np.concatenate([
    rng.normal(12, 2, 80),   # grupo de alfa baja
    rng.normal(22, 3, 40),   # grupo de alfa alta
])
```

4. *(Desafío)* **Entropía.** La entropía diferencial de una gaussiana es $H = \frac{1}{2}\ln(2\pi e\sigma^2)$.  
   Demuestra numéricamente que, entre todas las distribuciones con varianza fija $\sigma^2$, la gaussiana maximiza la entropía.  
   Pista: compara $H$ con la entropía empírica de una Uniforme, Laplace y Student-t con la misma varianza.

## 📚 Conjuntos de datos utilizados / referenciados

| Conjunto de datos | Fuente | Notas |
|---|---|---|
| FC/PA simulados | `numpy.random` | Sintético — calibrado con rangos fisiológicos reales |
| PhysioNet MIT-BIH | https://physionet.org/content/mitdb/ | ECG real — se usa desde la Sesión 3 |
| DREAMER (EEG) | https://zenodo.org/record/546113 | Se usa en las Sesiones 3 y 5 |
| ENSANUT 2022 | https://ensanut.insp.mx | Prevalencias de DM2 y prediabetes en adultos mexicanos — usadas en Parte 3b |